# Phase 5: Top-10 Feature Sensitivity Analysis

**Purpose:** Address reviewer feedback by demonstrating that a CatBoost model restricted to the top 10 SHAP-ranked features achieves comparable performance to the full feature model.

**Inputs:**
- `../data/processed/03_cleaned.parquet` — Phase 3 cleaned dataset
- `tuning_results_20260426_180035/catboost_best_model.pkl` — tuned CatBoost (Model A)
- `tuning_results_20260426_180035/catboost_study.pkl` — Optuna study with best hyperparameters

**Outputs:**
- `../outputs/model_a_metrics.json` — full-model metrics with bootstrap 95% CIs
- `../outputs/model_b_metrics.json` — top-10 model metrics with bootstrap 95% CIs
- `../outputs/model_b_top10.pkl` — saved top-10 CatBoost model
- `../outputs/figure_1_roc.png`, `figure_2_metrics.png`, `figure_3_shap_perm.png`, `figure_4_shap_ranking.png`, `figure_5_shap_dependence.png`, `figure_7_shap_summary.png`
- `../outputs/khamron_summary.md` — summary text for the response letter

**Methodology:**
1. Reproduce Phase 4's preprocessing pipeline and train/val/test split (same seed)
2. Load saved tuned CatBoost as Model A; verify by re-evaluating on test set
3. Identify top 10 features by manuscript label → engineered column name
4. Train Model B on top-10 subset using Model A's tuned hyperparameters
5. Bootstrap 95% CIs (1000 iterations) for both models on test set
6. SHAP analysis on Model B
7. Generate manuscript-style figures

**Top 10 features (manuscript Figure 4):**
1. Wheezing/Whistling in Chest (Past Year)
2. Close Relative Had Asthma (family history)
3. General Health Condition
4. FEV1/FVC Ratio (engineered from spirometry)
5. Times Received Healthcare (Past Year)
6. Health Now vs. 1 Year Ago
7. Family History × Lung Function (engineered interaction)
8. Forced Expiratory Time (seconds)
9. Crawl/Walk/Run/Play Limitations
10. Airway Obstruction Indicator (engineered binary)

In [1]:
"""
Phase 5 setup: imports, load saved Phase 4 tuned CatBoost and Optuna study.
"""

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paths
PROCESSED_DIR = Path("../data/processed")
OUTPUTS_DIR = Path("../outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
TUNING_DIR = Path("tuning_results_20260426_180035")

# Verify everything we need exists
required_files = [
    PROCESSED_DIR / "03_cleaned.parquet",
    TUNING_DIR / "catboost_best_model.pkl",
    TUNING_DIR / "catboost_study.pkl",
]
for f in required_files:
    if not f.exists():
        print(f"MISSING: {f}")
    else:
        print(f"  OK: {f} ({f.stat().st_size // 1024} KB)")

# Load the tuned model (Model A) and Optuna study
model_a = joblib.load(TUNING_DIR / "catboost_best_model.pkl")
study_catboost = joblib.load(TUNING_DIR / "catboost_study.pkl")

print()
print(f"Model A type: {type(model_a).__name__}")
print(f"Optuna study has {len(study_catboost.trials)} trials")
print(f"Best trial value (validation sensitivity): {study_catboost.best_value:.3f}")
print()
print("Best CatBoost hyperparameters from Optuna:")
for k, v in study_catboost.best_params.items():
    print(f"  {k}: {v}")

  OK: ..\data\processed\03_cleaned.parquet (580 KB)
  OK: tuning_results_20260426_180035\catboost_best_model.pkl (1231 KB)
  OK: tuning_results_20260426_180035\catboost_study.pkl (58 KB)

Model A type: Pipeline
Optuna study has 100 trials
Best trial value (validation sensitivity): 0.722

Best CatBoost hyperparameters from Optuna:
  iterations: 200
  learning_rate: 0.03409322921660775
  depth: 4
  l2_leaf_reg: 5.005601538786807
  border_count: 229
  bagging_temperature: 0.6823157114272929
  random_strength: 9.066256900126039


In [5]:
"""
Load Phase 4's saved preprocessed data — the exact data state Model A was trained on.

This includes:
- All four fitted preprocessing transformers (cleaner, feat_eng, imputer, scaler)
- The exact train/val/test splits
- Sample weights for each split
- The list of feature names after engineering
"""

preprocessed = joblib.load(TUNING_DIR / "preprocessed_data.pkl")

# Unpack what we need
cleaner = preprocessed['cleaner']
feat_eng = preprocessed['feat_eng']
imputer = preprocessed['imputer']
scaler = preprocessed['scaler']

X_train = preprocessed['X_train']
y_train = preprocessed['y_train']
X_val = preprocessed['X_val']
y_val = preprocessed['y_val']
X_test = preprocessed['X_test']
y_test = preprocessed['y_test']

sw_train = preprocessed['sw_train']
sw_val = preprocessed['sw_val']
sw_test = preprocessed['sw_test']

feature_names = preprocessed['feature_names']

# Confirm we got everything aligned
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print()
print(f"y_train asthma prevalence: {np.mean(y_train):.1%}")
print(f"y_test asthma prevalence:  {np.mean(y_test):.1%}")
print()
print(f"Number of feature names: {len(feature_names)}")
print(f"First 10 feature names: {feature_names[:10]}")
print()
print(f"Last 5 feature names: {feature_names[-5:]}")

X_train: (3939, 112)
X_val:   (1314, 112)
X_test:  (1314, 112)

y_train asthma prevalence: 18.7%
y_test asthma prevalence:  18.7%

Number of feature names: 112
First 10 feature names: ['RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDAGEEX', 'RIDRETH1', 'DMDBORN2', 'DMDCITZN', 'DMDEDUC3', 'DMDSCHOL', 'DMDHHSIZ']

Last 5 feature names: ['family_spirometry_interaction', 'SPXNFEV1_missing', 'SPXNFVC_missing', 'LBXCOT_missing', 'BMXBMI_missing']


In [6]:
"""
Map the manuscript's top 10 feature labels to actual column names in `feature_names`.

The manuscript reports labels like "Wheezing/Whistling in Chest (Past Year)" but our
feature matrix uses NHANES variable codes (RDQ070) and engineered feature names
(fev1_fvc_ratio, airway_obstruction, etc.).

This cell prints all 112 feature names so we can find the top 10 by inspection.
"""

# Print all feature names with their indices for reference
print("All 112 feature names:")
print("=" * 60)
for i, name in enumerate(feature_names):
    print(f"  {i:3d}: {name}")

All 112 feature names:
    0: RIAGENDR
    1: RIDAGEYR
    2: RIDAGEMN
    3: RIDAGEEX
    4: RIDRETH1
    5: DMDBORN2
    6: DMDCITZN
    7: DMDEDUC3
    8: DMDSCHOL
    9: DMDHHSIZ
   10: DMDFMSIZ
   11: INDHHIN2
   12: INDFMIN2
   13: INDFMPIR
   14: DMDHRGND
   15: DMDHRAGE
   16: DMDHRBR2
   17: DMDHREDU
   18: DMDHRMAR
   19: DMDHSEDU
   20: FIALANG
   21: WTINT2YR
   22: SDMVPSU
   23: SDMVSTRA
   24: BMXWT
   25: BMXHT
   26: BMXBMI
   27: BMXLEG
   28: BMXARML
   29: BMXARMC
   30: BMXWAIST
   31: BMXTRI
   32: BMXSUB
   33: MCQ053
   34: MCQ092
   35: MCQ140
   36: MCQ300B
   37: ENQ010
   38: ENQ020
   39: SPQ010
   40: SPQ020
   41: SPQ040
   42: SPQ060
   43: SPQ100
   44: ENQ100
   45: SPXNSTAT
   46: SPXNFVC
   47: SPXNEV
   48: SPXNFEV5
   49: SPXNFEV7
   50: SPXNFEV1
   51: SPXNFEV3
   52: SPXNFEV6
   53: SPXNPEF
   54: SPXNF257
   55: SPXNFET
   56: SPDNACC
   57: SPDBRONC
   58: LBXCOT
   59: LBDCOTLC
   60: URXNAL
   61: URDNALLC
   62: URXUCR
   63: FSD032A
   64: 

In [7]:
"""
Train Model B: CatBoost restricted to the top 10 SHAP-ranked features from the
manuscript, using the same tuned hyperparameters as Model A.

This is the sensitivity analysis — the question being: does removing all features
beyond the top 10 substantially change model performance?
"""

from catboost import CatBoostClassifier

# Top 10 features mapped from manuscript Figure 4 to actual column names
TOP_10_FEATURES = [
    'RDQ070',                            # 1. Wheezing/Whistling in Chest (Past Year)
    'MCQ300B',                           # 2. Close Relative Had Asthma
    'HUQ010',                            # 3. General Health Condition
    'fev1_fvc_ratio',                    # 4. FEV1/FVC Ratio
    'HUQ050',                            # 5. Times Received Healthcare (Past Year)
    'HSQ500',                            # 6. Health Now vs. 1 Year Ago
    'family_spirometry_interaction',     # 7. Family History × Lung Function
    'SPXNFET',                           # 8. Forced Expiratory Time (seconds)
    'PFQ020',                            # 9. Crawl/Walk/Run/Play Limitations
    'obstruction_indicator',             # 10. Airway Obstruction Indicator
]

# Verify all 10 are in feature_names
missing = [f for f in TOP_10_FEATURES if f not in feature_names]
if missing:
    print(f"❌ MISSING: {missing}")
    raise ValueError(f"Top-10 features missing from preprocessed data: {missing}")
else:
    print(f"✓ All 10 features confirmed in feature_names list")

# Get column indices for slicing the numpy arrays
top_10_indices = [feature_names.index(f) for f in TOP_10_FEATURES]
print(f"  Indices: {top_10_indices}")

# Slice train/val/test to top-10 features only
X_train_top10 = X_train[:, top_10_indices]
X_val_top10 = X_val[:, top_10_indices]
X_test_top10 = X_test[:, top_10_indices]

print()
print(f"X_train_top10: {X_train_top10.shape}")
print(f"X_val_top10:   {X_val_top10.shape}")
print(f"X_test_top10:  {X_test_top10.shape}")

# Phase 4's tuned hyperparameters (loaded from study_catboost.best_params)
best_params = study_catboost.best_params
print()
print(f"Using Phase 4's tuned hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# Configure CatBoost with the same tuned settings
catboost_params = {
    **best_params,
    'random_state': RANDOM_STATE,
    'verbose': False,
    'thread_count': -1,
}

# Train Model B
print()
print("Training Model B (CatBoost on top 10 features)...")
import time
start = time.time()
model_b = CatBoostClassifier(**catboost_params)
model_b.fit(X_train_top10, y_train, sample_weight=sw_train)
elapsed = time.time() - start
print(f"✓ Trained in {elapsed:.1f}s")

# Save the model
joblib.dump(model_b, OUTPUTS_DIR / "model_b_top10.pkl")
print(f"✓ Saved to: {OUTPUTS_DIR / 'model_b_top10.pkl'}")

✓ All 10 features confirmed in feature_names list
  Indices: [90, 36, 81, 96, 85, 78, 107, 55, 88, 97]

X_train_top10: (3939, 10)
X_val_top10:   (1314, 10)
X_test_top10:  (1314, 10)

Using Phase 4's tuned hyperparameters:
  iterations: 200
  learning_rate: 0.03409322921660775
  depth: 4
  l2_leaf_reg: 5.005601538786807
  border_count: 229
  bagging_temperature: 0.6823157114272929
  random_strength: 9.066256900126039

Training Model B (CatBoost on top 10 features)...
✓ Trained in 0.3s
✓ Saved to: ..\outputs\model_b_top10.pkl


In [8]:
print(f"Model B exists: {model_b is not None}")
print(f"Model B has {model_b.tree_count_} trees")
print(f"X_train_top10 still in memory: shape {X_train_top10.shape}")

Model B exists: True
Model B has 200 trees
X_train_top10 still in memory: shape (3939, 10)


In [12]:
"""
Cell 6: Evaluate Model A (full features) and Model B (top 10) on the held-out test set.
Compute AUC, sensitivity, specificity, PPV, NPV, F1, MCC, Brier with bootstrap 95% CIs.

Both models share the same X_test/y_test, so this is a direct comparison.
Model A uses the full 112-feature X_test; Model B uses X_test_top10 (10 features).

This version (v2) converts pandas Series to numpy arrays before bootstrap to ensure
positional indexing works correctly with current pandas. Also uses per-metric
resilience so a single failure doesn't drop the whole resample.
"""

from sklearn.metrics import (
    roc_auc_score, confusion_matrix, f1_score,
    matthews_corrcoef, brier_score_loss
)
import time

N_BOOTSTRAP = 1000  # iterations for 95% CIs
THRESHOLD = 0.5     # decision threshold for binary predictions

# Get prediction probabilities for both models on test set
print("Generating predictions...")
y_proba_a = model_a.predict_proba(X_test)[:, 1]
y_proba_b = model_b.predict_proba(X_test_top10)[:, 1]

# Apply threshold to get binary predictions
y_pred_a = (y_proba_a >= THRESHOLD).astype(int)
y_pred_b = (y_proba_b >= THRESHOLD).astype(int)

print(f"  Model A predictions on {len(y_test):,} test samples")
print(f"  Model B predictions on {len(y_test):,} test samples")
print()


def compute_metrics(y_true, y_pred, y_proba):
    """Returns dict of all metrics for one model."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    return {
        'AUC':         roc_auc_score(y_true, y_proba),
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'PPV':         ppv,
        'NPV':         npv,
        'F1':          f1_score(y_true, y_pred),
        'MCC':         matthews_corrcoef(y_true, y_pred),
        'Brier':       brier_score_loss(y_true, y_proba),
    }


def bootstrap_ci(y_true, y_pred, y_proba, n_iterations=1000, seed=42):
    """Compute 95% CIs for all metrics via bootstrap resampling.

    Inputs must be numpy arrays (not pandas Series) for positional indexing.
    Each metric is computed independently so a failure in one doesn't drop the
    whole resample. Resamples with only one class are skipped (AUC undefined).
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    metric_keys = ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1', 'MCC', 'Brier']
    samples = {k: [] for k in metric_keys}

    for i in range(n_iterations):
        idx = rng.choice(n, size=n, replace=True)
        yt, yp, ypr = y_true[idx], y_pred[idx], y_proba[idx]

        # Skip resamples with only one class — AUC undefined
        if len(np.unique(yt)) < 2:
            continue

        # Compute each metric independently
        try: samples['AUC'].append(roc_auc_score(yt, ypr))
        except Exception: pass

        try:
            tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
            samples['Sensitivity'].append(tp / (tp + fn) if (tp + fn) > 0 else 0)
            samples['Specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)
            samples['PPV'].append(tp / (tp + fp) if (tp + fp) > 0 else 0)
            samples['NPV'].append(tn / (tn + fn) if (tn + fn) > 0 else 0)
        except Exception: pass

        try: samples['F1'].append(f1_score(yt, yp))
        except Exception: pass

        try: samples['MCC'].append(matthews_corrcoef(yt, yp))
        except Exception: pass

        try: samples['Brier'].append(brier_score_loss(yt, ypr))
        except Exception: pass

    cis = {}
    for k in metric_keys:
        arr = np.array(samples[k])
        if len(arr) == 0:
            cis[k] = (float('nan'), float('nan'))
            print(f"  ⚠️  {k}: 0 valid resamples, returning NaN CI")
        else:
            cis[k] = (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))
    return cis


# Compute point metrics
metrics_a = compute_metrics(y_test, y_pred_a, y_proba_a)
metrics_b = compute_metrics(y_test, y_pred_b, y_proba_b)

# Convert to numpy arrays for positional bootstrap indexing
# (avoids pandas-3 label-based indexing issues)
y_test_arr = np.asarray(y_test)
y_pred_a_arr = np.asarray(y_pred_a)
y_pred_b_arr = np.asarray(y_pred_b)
y_proba_a_arr = np.asarray(y_proba_a)
y_proba_b_arr = np.asarray(y_proba_b)

# Bootstrap CIs (this is the slow part)
print(f"Bootstrapping {N_BOOTSTRAP} resamples per model...")
start = time.time()
cis_a = bootstrap_ci(y_test_arr, y_pred_a_arr, y_proba_a_arr, n_iterations=N_BOOTSTRAP)
print(f"  Model A done ({time.time()-start:.1f}s)")
start = time.time()
cis_b = bootstrap_ci(y_test_arr, y_pred_b_arr, y_proba_b_arr, n_iterations=N_BOOTSTRAP)
print(f"  Model B done ({time.time()-start:.1f}s)")

# Print comparison table
print()
print("=" * 78)
print(f"{'Metric':<14} {'Model A (full, 112 features)':<32} {'Model B (top 10 features)':<32}")
print("-" * 78)
for metric in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1', 'MCC', 'Brier']:
    a_val = metrics_a[metric]
    a_lo, a_hi = cis_a[metric]
    b_val = metrics_b[metric]
    b_lo, b_hi = cis_b[metric]
    print(f"{metric:<14} {a_val:.3f} ({a_lo:.3f}, {a_hi:.3f})         {b_val:.3f} ({b_lo:.3f}, {b_hi:.3f})")
print("=" * 78)

# Save metrics to JSON for downstream use
results = {
    'model_a_full_features': {
        'point_estimates': metrics_a,
        'confidence_intervals_95': {k: list(v) for k, v in cis_a.items()},
        'n_features': 112,
        'n_test_samples': int(len(y_test)),
    },
    'model_b_top10_features': {
        'point_estimates': metrics_b,
        'confidence_intervals_95': {k: list(v) for k, v in cis_b.items()},
        'n_features': 10,
        'feature_list': TOP_10_FEATURES,
        'n_test_samples': int(len(y_test)),
    },
    'comparison_setup': {
        'same_train_test_split': True,
        'same_random_seed': RANDOM_STATE,
        'same_hyperparameters': True,
        'hyperparameters': dict(best_params),
        'bootstrap_iterations': N_BOOTSTRAP,
    },
}
with open(OUTPUTS_DIR / "model_comparison_metrics.json", 'w') as f:
    json.dump(results, f, indent=2)
print()
print(f"✓ Saved: {OUTPUTS_DIR / 'model_comparison_metrics.json'}")


Generating predictions...
  Model A predictions on 1,314 test samples
  Model B predictions on 1,314 test samples

Bootstrapping 1000 resamples per model...
  Model A done (3.0s)
  Model B done (2.9s)

Metric         Model A (full, 112 features)     Model B (top 10 features)       
------------------------------------------------------------------------------
AUC            0.822 (0.790, 0.851)         0.811 (0.779, 0.840)
Sensitivity    0.756 (0.700, 0.809)         0.293 (0.235, 0.349)
Specificity    0.742 (0.714, 0.767)         0.982 (0.974, 0.990)
PPV            0.403 (0.361, 0.445)         0.791 (0.707, 0.871)
NPV            0.930 (0.911, 0.947)         0.858 (0.838, 0.877)
F1             0.525 (0.483, 0.569)         0.427 (0.358, 0.490)
MCC            0.407 (0.357, 0.456)         0.422 (0.356, 0.486)
Brier          0.187 (0.176, 0.200)         0.113 (0.102, 0.124)

✓ Saved: ..\outputs\model_comparison_metrics.json


In [13]:
"""
Threshold tuning: find the decision threshold that achieves sensitivity >= 80%
with the highest possible specificity, using the validation set. Then evaluate
both models at their tuned thresholds on the test set.

This matches the methodology of the original published analysis, which constrained
CatBoost to sensitivity >= 80% (the screening operating point).

Both models tuned the same way ensures fair head-to-head comparison.
"""

from sklearn.metrics import roc_curve

TARGET_SENSITIVITY = 0.80


def find_threshold_for_sensitivity(y_true, y_proba, target_sens=0.80):
    """Find the highest threshold whose sensitivity >= target_sens on the given set.
    Returns (threshold, achieved_sensitivity, achieved_specificity)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    # tpr = sensitivity. Find thresholds where tpr >= target.
    valid = tpr >= target_sens
    if not valid.any():
        # Can't achieve target — return the highest-sensitivity threshold available
        best_idx = np.argmax(tpr)
    else:
        # Among thresholds that meet the constraint, pick the one with highest threshold
        # (= lowest fpr = highest specificity)
        valid_indices = np.where(valid)[0]
        best_idx = valid_indices[np.argmax(thresholds[valid_indices])]
    return thresholds[best_idx], tpr[best_idx], 1 - fpr[best_idx]


# Tune Model A on validation
y_proba_a_val = model_a.predict_proba(X_val)[:, 1]
threshold_a, sens_a_val, spec_a_val = find_threshold_for_sensitivity(
    np.asarray(y_val), y_proba_a_val, target_sens=TARGET_SENSITIVITY
)

# Tune Model B on validation
y_proba_b_val = model_b.predict_proba(X_val_top10)[:, 1]
threshold_b, sens_b_val, spec_b_val = find_threshold_for_sensitivity(
    np.asarray(y_val), y_proba_b_val, target_sens=TARGET_SENSITIVITY
)

print("Threshold tuning on VALIDATION set:")
print(f"  Model A: threshold={threshold_a:.4f}  → sens={sens_a_val:.3f}, spec={spec_a_val:.3f}")
print(f"  Model B: threshold={threshold_b:.4f}  → sens={sens_b_val:.3f}, spec={spec_b_val:.3f}")
print()

# Apply tuned thresholds to TEST set predictions
y_pred_a_tuned = (y_proba_a >= threshold_a).astype(int)
y_pred_b_tuned = (y_proba_b >= threshold_b).astype(int)

# Compute new point metrics
metrics_a_tuned = compute_metrics(y_test, y_pred_a_tuned, y_proba_a)
metrics_b_tuned = compute_metrics(y_test, y_pred_b_tuned, y_proba_b)

# Bootstrap CIs at tuned thresholds
y_pred_a_tuned_arr = np.asarray(y_pred_a_tuned)
y_pred_b_tuned_arr = np.asarray(y_pred_b_tuned)

print("Bootstrapping at tuned thresholds...")
start = time.time()
cis_a_tuned = bootstrap_ci(y_test_arr, y_pred_a_tuned_arr, y_proba_a_arr, n_iterations=N_BOOTSTRAP)
print(f"  Model A done ({time.time()-start:.1f}s)")
start = time.time()
cis_b_tuned = bootstrap_ci(y_test_arr, y_pred_b_tuned_arr, y_proba_b_arr, n_iterations=N_BOOTSTRAP)
print(f"  Model B done ({time.time()-start:.1f}s)")

# Print comparison table
print()
print("=" * 80)
print(f"TEST SET PERFORMANCE AT TUNED THRESHOLDS (sensitivity target ≥ {TARGET_SENSITIVITY:.0%})")
print("=" * 80)
print(f"{'Metric':<14} {'Model A (full, 112 features)':<32} {'Model B (top 10 features)':<32}")
print("-" * 80)
for metric in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1', 'MCC', 'Brier']:
    a_val = metrics_a_tuned[metric]
    a_lo, a_hi = cis_a_tuned[metric]
    b_val = metrics_b_tuned[metric]
    b_lo, b_hi = cis_b_tuned[metric]
    print(f"{metric:<14} {a_val:.3f} ({a_lo:.3f}, {a_hi:.3f})         {b_val:.3f} ({b_lo:.3f}, {b_hi:.3f})")
print("=" * 80)

# Update the saved JSON with tuned-threshold results
results['model_a_full_features']['tuned_threshold'] = {
    'threshold': float(threshold_a),
    'target_sensitivity': TARGET_SENSITIVITY,
    'point_estimates': metrics_a_tuned,
    'confidence_intervals_95': {k: list(v) for k, v in cis_a_tuned.items()},
}
results['model_b_top10_features']['tuned_threshold'] = {
    'threshold': float(threshold_b),
    'target_sensitivity': TARGET_SENSITIVITY,
    'point_estimates': metrics_b_tuned,
    'confidence_intervals_95': {k: list(v) for k, v in cis_b_tuned.items()},
}
with open(OUTPUTS_DIR / "model_comparison_metrics.json", 'w') as f:
    json.dump(results, f, indent=2)
print()
print(f"✓ Updated: {OUTPUTS_DIR / 'model_comparison_metrics.json'}")

Threshold tuning on VALIDATION set:
  Model A: threshold=0.4740  → sens=0.801, spec=0.710
  Model B: threshold=0.1586  → sens=0.801, spec=0.676

Bootstrapping at tuned thresholds...
  Model A done (3.0s)
  Model B done (2.9s)

TEST SET PERFORMANCE AT TUNED THRESHOLDS (sensitivity target ≥ 80%)
Metric         Model A (full, 112 features)     Model B (top 10 features)       
--------------------------------------------------------------------------------
AUC            0.822 (0.790, 0.851)         0.811 (0.779, 0.840)
Sensitivity    0.780 (0.726, 0.829)         0.772 (0.720, 0.820)
Specificity    0.711 (0.683, 0.737)         0.681 (0.650, 0.708)
PPV            0.383 (0.341, 0.427)         0.358 (0.318, 0.396)
NPV            0.934 (0.916, 0.950)         0.928 (0.910, 0.945)
F1             0.514 (0.470, 0.556)         0.489 (0.446, 0.528)
MCC            0.394 (0.347, 0.441)         0.360 (0.309, 0.404)
Brier          0.187 (0.176, 0.200)         0.113 (0.102, 0.124)

✓ Updated: ..\outputs\